# CIC-IDS-2017 data exploration

This notebook validates the merged CIC-IDS-2017 CSV before preprocessing or model training. The project-added `ClassLabel` column has been removed from the CSV; `Label` remains the source-of-truth target.

## Dataset reference

Static CICFlowMeter terminology and the complete feature dictionary are maintained in [the dataset reference](../DATASET.md). This notebook focuses on the observed data-quality investigation, cleaning evidence, and resulting dataset.

## 1. Imports and dataset path

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

dataset_candidates = [
    Path("../data/raw/cicids2017_merged.csv"),
    Path("ml/data/raw/cicids2017_merged.csv"),
]

DATASET_PATH = next(
    (path.resolve() for path in dataset_candidates if path.exists()),
    None,
)

if DATASET_PATH is None:
    raise FileNotFoundError(
        "Could not find ml/data/raw/cicids2017_merged.csv. "
        "Run the notebook from the repository root or ml/notebooks."
    )

print(f"Dataset: {DATASET_PATH}")
print(f"File size: {DATASET_PATH.stat().st_size / (1024 ** 3):.2f} GiB")

Dataset: C:\Users\ademz\Desktop\9raya\DoS-Intrusion-Detection-System\ml\data\raw\cicids2017_merged.csv
File size: 1.45 GiB


## 2. Load the dataset

The merged CSV is large, so this cell may take some time and require several gigabytes of memory.

In [2]:
df = pd.read_csv(DATASET_PATH, low_memory=False)

if "ClassLabel" in df.columns:
    raise ValueError("ClassLabel should not be present in the raw dataset.")

print("Dataset loaded successfully.")

Dataset loaded successfully.


## 3. Dataset dimensions and columns

In [3]:
row_count, column_count = df.shape

print(f"Rows: {row_count:,}")
print(f"Columns: {column_count}")
print("\nColumn names:")

for index, column in enumerate(df.columns, start=1):
    print(f"{index}. {column}")

Rows: 3,119,345
Columns: 84

Column names:
1. Flow ID
2. Src IP
3. Src Port
4. Dst IP
5. Dst Port
6. Protocol
7. Timestamp
8. Flow Duration
9. Total Fwd Packet
10. Total Bwd packets
11. Total Length of Fwd Packet
12. Total Length of Bwd Packet
13. Fwd Packet Length Max
14. Fwd Packet Length Min
15. Fwd Packet Length Mean
16. Fwd Packet Length Std
17. Bwd Packet Length Max
18. Bwd Packet Length Min
19. Bwd Packet Length Mean
20. Bwd Packet Length Std
21. Flow Bytes/s
22. Flow Packets/s
23. Flow IAT Mean
24. Flow IAT Std
25. Flow IAT Max
26. Flow IAT Min
27. Fwd IAT Total
28. Fwd IAT Mean
29. Fwd IAT Std
30. Fwd IAT Max
31. Fwd IAT Min
32. Bwd IAT Total
33. Bwd IAT Mean
34. Bwd IAT Std
35. Bwd IAT Max
36. Bwd IAT Min
37. Fwd PSH Flags
38. Bwd PSH Flags
39. Fwd URG Flags
40. Bwd URG Flags
41. Fwd Header Length
42. Bwd Header Length
43. Fwd Packets/s
44. Bwd Packets/s
45. Packet Length Min
46. Packet Length Max
47. Packet Length Mean
48. Packet Length Std
49. Packet Length Variance
50. FIN

## 4. Duplicate column names

Check this before selecting or transforming columns by name.

In [4]:
duplicate_columns = df.columns[df.columns.duplicated()].tolist()

print(f"Duplicate column names: {len(duplicate_columns)}")
print(duplicate_columns if duplicate_columns else "No duplicate column names found.")

Duplicate column names: 0
No duplicate column names found.


## 5. First rows

In [5]:
display(df.head())

,Flow ID,Src IP,Src Port,Dst IP,Dst Port,Protocol,Timestamp,Flow Duration,Total Fwd Packet,Total Bwd packets,...,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,192.168.10.5-104.16.207.165-54865-443-6,104.16.207.165,443.0,192.168.10.5,54865.0,6.0,7/7/2017 3:30,3.0,2.0,0.0,...,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
1,192.168.10.5-104.16.28.216-55054-80-6,104.16.28.216,80.0,192.168.10.5,55054.0,6.0,7/7/2017 3:30,109.0,1.0,1.0,...,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
2,192.168.10.5-104.16.28.216-55055-80-6,104.16.28.216,80.0,192.168.10.5,55055.0,6.0,7/7/2017 3:30,52.0,1.0,1.0,...,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
3,192.168.10.16-104.17.241.25-46236-443-6,104.17.241.25,443.0,192.168.10.16,46236.0,6.0,7/7/2017 3:30,34.0,1.0,1.0,...,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
4,192.168.10.5-104.19.196.102-54863-443-6,104.19.196.102,443.0,192.168.10.5,54863.0,6.0,7/7/2017 3:30,3.0,2.0,0.0,...,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN


## 6. Data types and memory usage

In [6]:
df.info(memory_usage="deep")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3119345 entries, 0 to 3119344
Data columns (total 84 columns):
 #   Column                      Dtype  
---  ------                      -----  
 0   Flow ID                     object 
 1   Src IP                      object 
 2   Src Port                    float64
 3   Dst IP                      object 
 4   Dst Port                    float64
 5   Protocol                    float64
 6   Timestamp                   object 
 7   Flow Duration               float64
 8   Total Fwd Packet            float64
 9   Total Bwd packets           float64
 10  Total Length of Fwd Packet  float64
 11  Total Length of Bwd Packet  float64
 12  Fwd Packet Length Max       float64
 13  Fwd Packet Length Min       float64
 14  Fwd Packet Length Mean      float64
 15  Fwd Packet Length Std       float64
 16  Bwd Packet Length Max       float64
 17  Bwd Packet Length Min       float64
 18  Bwd Packet Length Mean      float64
 19  Bwd Packet Length Std

## 7. Validate the target label

A flow without `Label` cannot be used for supervised learning. Check missing labels first and confirm whether those rows contain any useful feature values.

In [7]:
invalid_label_mask = df["Label"].isna()
invalid_row_count = int(invalid_label_mask.sum())
invalid_row_percentage = invalid_row_count / len(df) * 100

print(f"Rows without Label: {invalid_row_count:,} ({invalid_row_percentage:.2f}%)")
display(df.loc[invalid_label_mask].head())

invalid_non_null_counts = df.loc[invalid_label_mask].notna().sum()
invalid_non_null_counts = invalid_non_null_counts[
    invalid_non_null_counts > 0
].sort_values(ascending=False)

if invalid_non_null_counts.empty:
    print("All rows without Label are completely empty across the 84 columns.")
else:
    print("Non-null values found in rows without Label:")
    display(invalid_non_null_counts.to_frame("non_null_count"))

Rows without Label: 288,602 (9.25%)


,Flow ID,Src IP,Src Port,Dst IP,Dst Port,Protocol,Timestamp,Flow Duration,Total Fwd Packet,Total Bwd packets,...,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
1692131,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1692132,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1692133,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1692134,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1692135,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


All rows without Label are completely empty across the 84 columns.


## 8. Create the valid working dataset

This changes only the in-memory DataFrame. It does not modify the CSV.

In [8]:
rows_before = len(df)
df.dropna(subset=["Label"], inplace=True)
df.reset_index(drop=True, inplace=True)
df_valid = df
del df
del invalid_label_mask

print(f"Rows before: {rows_before:,}")
print(f"Rows removed: {rows_before - len(df_valid):,}")
print(f"Valid rows remaining: {len(df_valid):,}")

Rows before: 3,119,345
Rows removed: 288,602
Valid rows remaining: 2,830,743


## 9. Detailed label distribution

Inspect the target classes only after removing rows that have no usable target.

In [9]:
label_counts = df_valid["Label"].value_counts(dropna=False)
label_distribution = pd.DataFrame(
    {
        "count": label_counts,
        "percentage": (label_counts / len(df_valid) * 100).round(6),
    }
)

print(f"Unique detailed labels: {df_valid['Label'].nunique(dropna=False)}")
display(label_distribution)

Unique detailed labels: 15


,count,percentage
Label,,
BENIGN,2273097,80.300366
DoS Hulk,231073,8.162981
PortScan,158930,5.614427
DDoS,128027,4.522735
DoS GoldenEye,10293,0.363615
FTP-Patator,7938,0.280421
SSH-Patator,5897,0.208320
DoS slowloris,5796,0.204752
DoS Slowhttptest,5499,0.194260


## 10. Normalize malformed label text

Standardize the malformed separator in the three `Web Attack` labels. This changes only their spelling, not their class membership or row counts.

In [10]:
web_attack_mask = df_valid["Label"].str.startswith("Web Attack", na=False)
web_attack_counts_before = (
    df_valid.loc[web_attack_mask, "Label"].value_counts()
)

normalized_web_attack_labels_by_suffix = {
    "Brute Force": "Web Attack - Brute Force",
    "XSS": "Web Attack - XSS",
    "Sql Injection": "Web Attack - Sql Injection",
}

web_attack_label_replacements = {}
for original_label in web_attack_counts_before.index:
    matching_normalized_labels = [
        normalized_label
        for suffix, normalized_label
        in normalized_web_attack_labels_by_suffix.items()
        if str(original_label).endswith(suffix)
    ]
    if len(matching_normalized_labels) != 1:
        raise ValueError(
            f"Unrecognized Web Attack label: {original_label!r}"
        )
    web_attack_label_replacements[original_label] = (
        matching_normalized_labels[0]
    )

label_normalization_summary = pd.DataFrame(
    {
        "Original label": list(web_attack_label_replacements),
        "Normalized label": list(
            web_attack_label_replacements.values()
        ),
        "Rows affected": [
            int(web_attack_counts_before[label])
            for label in web_attack_label_replacements
        ],
    }
)

df_valid["Label"] = df_valid["Label"].replace(
    web_attack_label_replacements
)

normalized_web_attack_labels = set(
    df_valid.loc[
        df_valid["Label"].str.startswith("Web Attack", na=False),
        "Label",
    ].unique()
)
expected_web_attack_labels = set(
    normalized_web_attack_labels_by_suffix.values()
)
if normalized_web_attack_labels != expected_web_attack_labels:
    raise AssertionError("Web Attack label normalization failed.")

display(label_normalization_summary)
print("Web Attack labels after normalization:")
for label in sorted(normalized_web_attack_labels):
    print(f"- {label}")

,Original label,Normalized label,Rows affected
0,Web Attack  Brute Force,Web Attack - Brute Force,1507
1,Web Attack  XSS,Web Attack - XSS,652
2,Web Attack  Sql Injection,Web Attack - Sql Injection,21


Web Attack labels after normalization:
- Web Attack - Brute Force
- Web Attack - Sql Injection
- Web Attack - XSS


All three `Web Attack` classes are preserved with the same row counts. Their separator is now a plain hyphen, avoiding encoding-dependent text such as `Â–` or an invalid control character in later tables and target encoders.

## 11. Missing feature values before infinity normalization

This records the values that were originally missing, before infinity is converted to `NaN`.

In [11]:
feature_columns = df_valid.columns.drop("Label")
missing_before = df_valid.loc[:, feature_columns].isna().sum()
missing_before_summary = pd.DataFrame(
    {
        "missing_count": missing_before,
        "missing_percentage": (missing_before / len(df_valid) * 100).round(6),
    }
).sort_values("missing_count", ascending=False)

print(f"Originally missing feature values: {int(missing_before.sum()):,}")
display(missing_before_summary[missing_before_summary["missing_count"] > 0])

Originally missing feature values: 1,358


,missing_count,missing_percentage
Flow Bytes/s,1358,0.047973


## 12. Inspect infinite feature values

Count infinite values without changing the dataset.

In [12]:
numeric_columns = df_valid.select_dtypes(include="number").columns
infinite_counts = pd.Series(
    {
        column: int(np.isinf(df_valid[column].to_numpy()).sum())
        for column in numeric_columns
    },
    name="infinite_count",
).sort_values(ascending=False)

infinite_counts = infinite_counts[infinite_counts > 0]
print(f"Columns containing infinity: {len(infinite_counts)}")
print(f"Total infinite values: {int(infinite_counts.sum()):,}")
display(infinite_counts.to_frame())

Columns containing infinity: 2
Total infinite values: 4,376


,infinite_count
Flow Packets/s,2867
Flow Bytes/s,1509


## 13. Normalize infinite values

Convert positive and negative infinity to `NaN` so every non-finite feature value has one consistent representation.

In [13]:
affected_columns = infinite_counts.index.tolist()

if affected_columns:
    df_valid.loc[:, affected_columns] = df_valid[affected_columns].replace(
        [np.inf, -np.inf],
        np.nan,
    )

print(f"Infinite values converted to NaN: {int(infinite_counts.sum()):,}")

Infinite values converted to NaN: 4,376


## 14. Missing feature values after infinity normalization

This is the final missing-value pool that preprocessing must handle.

In [14]:
missing_after = df_valid.loc[:, feature_columns].isna().sum()
missing_after_summary = pd.DataFrame(
    {
        "missing_count": missing_after,
        "missing_percentage": (missing_after / len(df_valid) * 100).round(6),
    }
).sort_values("missing_count", ascending=False)

print(f"Missing feature values after normalization: {int(missing_after.sum()):,}")
display(missing_after_summary[missing_after_summary["missing_count"] > 0])

Missing feature values after normalization: 5,734


,missing_count,missing_percentage
Flow Bytes/s,2867,0.101281
Flow Packets/s,2867,0.101281


## 15. Numeric feature descriptive statistics

Inspect distributions after infinity normalization. Transposing the output makes the feature-level summary easier to read.

In [15]:
numeric_summary = df_valid.loc[:, numeric_columns].describe(
    percentiles=[0.01, 0.25, 0.50, 0.75, 0.99]
).T

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.width", None,
):
    display(numeric_summary)

,count,mean,std,min,1%,25%,50%,75%,99%,max
Src Port,2830743.0,4.112886e+04,2.229494e+04,0.000000e+00,80.000000,32774.000000,50944.000000,5.841300e+04,6.490800e+04,6.553500e+04
Dst Port,2830743.0,8.071483e+03,1.828363e+04,0.000000e+00,22.000000,53.000000,80.000000,4.430000e+02,6.165200e+04,6.553500e+04
Protocol,2830743.0,9.880341e+00,5.261922e+00,0.000000e+00,6.000000,6.000000,6.000000,1.700000e+01,1.700000e+01,1.700000e+01
Flow Duration,2830743.0,1.478566e+07,3.365374e+07,-1.300000e+01,1.000000,155.000000,31316.000000,3.204828e+06,1.177776e+08,1.200000e+08
Total Fwd Packet,2830743.0,9.361160e+00,7.496728e+02,1.000000e+00,1.000000,2.000000,2.000000,5.000000e+00,4.800000e+01,2.197590e+05
Total Bwd packets,2830743.0,1.039377e+01,9.973883e+02,0.000000e+00,0.000000,1.000000,2.000000,4.000000e+00,5.700000e+01,2.919220e+05
Total Length of Fwd Packet,2830743.0,5.493024e+02,9.993589e+03,0.000000e+00,0.000000,12.000000,62.000000,1.870000e+02,1.159500e+04,1.290000e+07
Total Length of Bwd Packet,2830743.0,1.616264e+04,2.263088e+06,0.000000e+00,0.000000,0.000000,123.000000,4.820000e+02,7.183100e+04,6.554530e+08
Fwd Packet Length Max,2830743.0,2.075999e+02,7.171848e+02,0.000000e+00,0.000000,6.000000,37.000000,8.100000e+01,2.936000e+03,2.482000e+04
Fwd Packet Length Min,2830743.0,1.871366e+01,6.033935e+01,0.000000e+00,0.000000,0.000000,2.000000,3.600000e+01,7.700000e+01,2.325000e+03


## 16. Zero and negative flow durations

Zero duration can represent a single-packet or sub-resolution flow. Negative duration is invalid. Measure them separately and inspect their target classes.

In [16]:
zero_duration_mask = df_valid["Flow Duration"] == 0
negative_duration_mask = df_valid["Flow Duration"] < 0
zero_duration_count = int(zero_duration_mask.sum())
negative_duration_count = int(negative_duration_mask.sum())

duration_summary = pd.DataFrame(
    {
        "count": [zero_duration_count, negative_duration_count],
        "percentage": [
            zero_duration_mask.mean() * 100,
            negative_duration_mask.mean() * 100,
        ],
    },
    index=["zero_duration", "negative_duration"],
)
display(duration_summary)

for duration_type, mask in {
    "zero_duration": zero_duration_mask,
    "negative_duration": negative_duration_mask,
}.items():
    print(f"\nLabel distribution for {duration_type}:")
    display(
        df_valid.loc[mask, "Label"]
        .value_counts()
        .to_frame("flow_count")
    )

,count,percentage
zero_duration,2867,0.101281
negative_duration,115,0.004063



Label distribution for zero_duration:


,flow_count
Label,
BENIGN,1777
DoS Hulk,949
PortScan,126
Bot,10
FTP-Patator,3
DDoS,2



Label distribution for negative_duration:


,flow_count
Label,
BENIGN,115


## 17. Systematic negative-value and TCP-window validation

Every numeric feature should be non-negative except the two TCP-window fields, where `-1` is an unavailable/not-applicable sentinel. Check all other numeric features systematically, then validate the window fields separately: accepted values are `-1` or the unsigned 16-bit range `0..65535`.

In [17]:
window_sentinel_features = ["FWD Init Win Bytes", "Bwd Init Win Bytes"]
non_negative_features = [
    feature
    for feature in numeric_columns
    if feature not in window_sentinel_features
]

negative_counts = pd.Series(
    {
        feature: int((df_valid[feature] < 0).sum())
        for feature in non_negative_features
    },
    name="negative_count",
)
negative_counts = negative_counts[negative_counts > 0].sort_values(ascending=False)

if negative_counts.empty:
    print("No unexpected negative values found.")
else:
    negative_summary = pd.DataFrame(
        {
            "negative_count": negative_counts,
            "percentage": (negative_counts / len(df_valid) * 100).round(6),
            "minimum": [
                df_valid[feature].min()
                for feature in negative_counts.index
            ],
        }
    )
    display(negative_summary)

    negative_label_distribution = pd.concat(
        {
            feature: df_valid.loc[
                df_valid[feature] < 0,
                "Label",
            ].value_counts()
            for feature in negative_counts.index
        },
        names=["feature", "Label"],
    ).rename("count").to_frame()

    negative_label_distribution["percentage_within_feature"] = (
        negative_label_distribution["count"]
        / negative_label_distribution.groupby(level="feature")["count"].transform("sum")
        * 100
    ).round(4)

    with pd.option_context("display.max_rows", None):
        display(negative_label_distribution)

window_validation_records = []
invalid_window_label_records = []

for feature in window_sentinel_features:
    values = df_valid[feature]
    invalid_mask = (values < -1) | (values > 65535)
    invalid_count = int(invalid_mask.sum())

    window_validation_records.append(
        {
            "feature": feature,
            "minimum": values.min(),
            "maximum": values.max(),
            "sentinel_minus_one_count": int((values == -1).sum()),
            "invalid_count": invalid_count,
            "invalid_percentage": invalid_count / len(df_valid) * 100,
        }
    )

    if invalid_count:
        label_counts_for_invalid = df_valid.loc[
            invalid_mask,
            "Label",
        ].value_counts()
        for label, count in label_counts_for_invalid.items():
            invalid_window_label_records.append(
                {"feature": feature, "Label": label, "count": int(count)}
            )

window_validation_summary = pd.DataFrame(window_validation_records).set_index("feature")
display(window_validation_summary)

if invalid_window_label_records:
    display(pd.DataFrame(invalid_window_label_records))
else:
    print("No TCP-window values below -1 or above 65535.")

,negative_count,percentage,minimum
Flow IAT Min,2891,0.102129,-1.400000e+01
Flow Duration,115,0.004063,-1.300000e+01
Flow IAT Mean,115,0.004063,-1.300000e+01
Flow Packets/s,115,0.004063,-2.000000e+06
Flow IAT Max,115,0.004063,-1.300000e+01
Flow Bytes/s,85,0.003003,-2.610000e+08
Fwd Header Length,35,0.001236,-3.221223e+10
Fwd Seg Size Min,35,0.001236,-5.368707e+08
Bwd Header Length,22,0.000777,-1.073741e+09
Fwd IAT Min,17,0.000601,-1.200000e+01


count  percentage_within_feature
feature           Label                                          
Flow IAT Min      BENIGN          2697                    93.2895
                  DoS Hulk         159                     5.4998
                  DDoS              19                     0.6572
                  DoS GoldenEye      5                     0.1730
                  Heartbleed         4                     0.1384
                  FTP-Patator        4                     0.1384
                  SSH-Patator        2                     0.0692
                  Infiltration       1                     0.0346
Flow Duration     BENIGN           115                   100.0000
Flow IAT Mean     BENIGN           115                   100.0000
Flow Packets/s    BENIGN           115                   100.0000
Flow IAT Max      BENIGN           115                   100.0000
Flow Bytes/s      BENIGN            85                   100.0000
Fwd Header Length BENIGN            35                   100.0000
Fwd Seg Size Min  BENIGN            35                   100.0000
Bwd Header Length BENIGN            22                   100.0000
Fwd IAT Min       DoS Hulk           8                    47.0588
                  DDoS               6                    35.2941
                  DoS GoldenEye      3                    17.6471

,minimum,maximum,sentinel_minus_one_count,invalid_count,invalid_percentage
feature,,,,,
FWD Init Win Bytes,-1.0,65535.0,1001189,0,0.0
Bwd Init Win Bytes,-1.0,65535.0,1441552,0,0.0


No TCP-window values below -1 or above 65535.


## 18. Exact constant columns

Constant features contain no information for classification. Detect them exactly without deleting them.

In [18]:
constant_records = []

for feature in feature_columns:
    values = df_valid[feature]
    first_value = values.iloc[0]
    is_constant = (
        values.isna().all()
        if pd.isna(first_value)
        else values.eq(first_value).all()
    )

    if is_constant:
        constant_records.append(
            {"feature": feature, "constant_value": first_value}
        )

constant_columns = pd.DataFrame(constant_records)
print(f"Exact constant columns: {len(constant_columns)}")
display(constant_columns)

Exact constant columns: 8


,feature,constant_value
0,Bwd PSH Flags,0.0
1,Bwd URG Flags,0.0
2,Fwd Bytes/Bulk Avg,0.0
3,Fwd Packet/Bulk Avg,0.0
4,Fwd Bulk Rate Avg,0.0
5,Bwd Bytes/Bulk Avg,0.0
6,Bwd Packet/Bulk Avg,0.0
7,Bwd Bulk Rate Avg,0.0


## 19. Remove rows with impossible negative values

Remove a row if it has a negative flow duration, any negative IAT feature, or a negative header/segment-size feature. These measurements cannot be negative. First report the individual reasons and their overlap so the same row is counted only once.

In [19]:
iat_features = [
    feature
    for feature in df_valid.select_dtypes(include="number").columns
    if "IAT" in feature
]

header_segment_features = [
    "Fwd Header Length",
    "Bwd Header Length",
    "Fwd Segment Size Avg",
    "Bwd Segment Size Avg",
    "Fwd Seg Size Min",
]

negative_reason_masks = pd.DataFrame(
    {
        "negative_flow_duration": df_valid["Flow Duration"].lt(0),
        "negative_IAT": df_valid[iat_features].lt(0).any(axis=1),
        "negative_header_or_segment": (
            df_valid[header_segment_features].lt(0).any(axis=1)
        ),
    },
    index=df_valid.index,
)

negative_reason_summary = pd.DataFrame(
    {
        "row_count": negative_reason_masks.sum(),
        "percentage": negative_reason_masks.mean().mul(100),
    }
)
display(negative_reason_summary)

invalid_negative_mask = negative_reason_masks.any(axis=1)
negative_overlap_summary = (
    negative_reason_masks.loc[invalid_negative_mask]
    .value_counts()
    .rename("row_count")
    .reset_index()
)
print("Overlap between removal reasons:")
display(negative_overlap_summary)

removed_label_distribution = (
    df_valid.loc[invalid_negative_mask, "Label"]
    .value_counts()
    .rename("removed_rows")
    .to_frame()
)

rows_before_negative_removal = len(df_valid)
rows_removed_for_negative_values = int(invalid_negative_mask.sum())
df_valid.drop(index=df_valid.index[invalid_negative_mask], inplace=True)
df_valid.reset_index(drop=True, inplace=True)

print(f"Rows before removal: {rows_before_negative_removal:,}")
print(f"Unique rows removed: {rows_removed_for_negative_values:,}")
print(f"Rows remaining: {len(df_valid):,}")
display(removed_label_distribution)

,row_count,percentage
negative_flow_duration,115,0.004063
negative_IAT,2891,0.102129
negative_header_or_segment,35,0.001236


Overlap between removal reasons:


,negative_flow_duration,negative_IAT,negative_header_or_segment,row_count
0,False,True,False,2776
1,True,True,False,115
2,False,False,True,35


Rows before removal: 2,830,743
Unique rows removed: 2,926
Rows remaining: 2,827,817


,removed_rows
Label,
BENIGN,2732
DoS Hulk,159
DDoS,19
DoS GoldenEye,5
Heartbleed,4
FTP-Patator,4
SSH-Patator,2
Infiltration,1


## 20. Remove constant feature columns

Remove only the eight exact constant columns because they contain no information. Keep all nonconstant semantically related features until the later correlation and feature-selection stage.

In [20]:
constant_features_to_drop = [
    "Bwd PSH Flags",
    "Bwd URG Flags",
    "Fwd Bytes/Bulk Avg",
    "Fwd Packet/Bulk Avg",
    "Fwd Bulk Rate Avg",
    "Bwd Bytes/Bulk Avg",
    "Bwd Packet/Bulk Avg",
    "Bwd Bulk Rate Avg",
]

features_to_drop = constant_features_to_drop
missing_features = [
    feature for feature in features_to_drop if feature not in df_valid.columns
]

if missing_features:
    raise KeyError(f"Expected features are missing: {missing_features}")

columns_before_removal = df_valid.shape[1]
df_valid.drop(columns=features_to_drop, inplace=True)

print(f"Columns before removal: {columns_before_removal}")
print(f"Columns removed: {len(features_to_drop)}")
print(f"Columns remaining: {df_valid.shape[1]}")
display(pd.Series(features_to_drop, name="removed_feature").to_frame())

Columns before removal: 84
Columns removed: 8
Columns remaining: 76


,removed_feature
0,Bwd PSH Flags
1,Bwd URG Flags
2,Fwd Bytes/Bulk Avg
3,Fwd Packet/Bulk Avg
4,Fwd Bulk Rate Avg
5,Bwd Bytes/Bulk Avg
6,Bwd Packet/Bulk Avg
7,Bwd Bulk Rate Avg


## 21. Compare missing rate values with zero duration

`Flow Bytes/s` and `Flow Packets/s` are undefined when duration is zero. Test the masks directly instead of assuming that equal counts mean they contain the same rows.

In [21]:
zero_duration_mask = df_valid["Flow Duration"].eq(0)
flow_bytes_rate_missing_mask = df_valid["Flow Bytes/s"].isna()
flow_packets_rate_missing_mask = df_valid["Flow Packets/s"].isna()
both_rates_missing_mask = (
    flow_bytes_rate_missing_mask & flow_packets_rate_missing_mask
)
either_rate_missing_mask = (
    flow_bytes_rate_missing_mask | flow_packets_rate_missing_mask
)

rate_duration_comparison = pd.Series(
    {
        "zero_duration_rows": int(zero_duration_mask.sum()),
        "Flow Bytes/s missing": int(flow_bytes_rate_missing_mask.sum()),
        "Flow Packets/s missing": int(flow_packets_rate_missing_mask.sum()),
        "both rates missing": int(both_rates_missing_mask.sum()),
        "either rate missing": int(either_rate_missing_mask.sum()),
        "zero duration and both missing": int(
            (zero_duration_mask & both_rates_missing_mask).sum()
        ),
        "zero duration without both missing": int(
            (zero_duration_mask & ~both_rates_missing_mask).sum()
        ),
        "both missing outside zero duration": int(
            (~zero_duration_mask & both_rates_missing_mask).sum()
        ),
    },
    name="row_count",
).to_frame()
display(rate_duration_comparison)

print(
    "Zero duration exactly matches missing Flow Bytes/s: ",
    zero_duration_mask.equals(flow_bytes_rate_missing_mask),
)
print(
    "Zero duration exactly matches missing Flow Packets/s: ",
    zero_duration_mask.equals(flow_packets_rate_missing_mask),
)
print(
    "Zero duration exactly matches both rates missing: ",
    zero_duration_mask.equals(both_rates_missing_mask),
)

,row_count
zero_duration_rows,2866
Flow Bytes/s missing,2866
Flow Packets/s missing,2866
both rates missing,2866
either rate missing,2866
zero duration and both missing,2866
zero duration without both missing,0
both missing outside zero duration,0


Zero duration exactly matches missing Flow Bytes/s:  True
Zero duration exactly matches missing Flow Packets/s:  True
Zero duration exactly matches both rates missing:  True


## 22. Investigate whether zero-duration flows are internally logical

A zero-duration flow is not automatically corrupt: one or more packets can receive the same timestamp at the capture resolution. It becomes suspicious when it has no packets, many packets, or non-zero IAT measurements. Zero transferred payload bytes can still be valid for TCP control packets.

Protocol numbers commonly used here are `6` for TCP and `17` for UDP. Inspect packet and byte totals, all IAT features, protocol and label distributions, and the largest zero-duration flows before deciding whether to keep them.

In [22]:
zero_duration_flows = df_valid.loc[zero_duration_mask].copy()
zero_duration_flows["Total Packets"] = (
    zero_duration_flows["Total Fwd Packet"]
    + zero_duration_flows["Total Bwd packets"]
)
zero_duration_flows["Total Bytes"] = (
    zero_duration_flows["Total Length of Fwd Packet"]
    + zero_duration_flows["Total Length of Bwd Packet"]
)

print(f"Zero-duration rows under investigation: {len(zero_duration_flows):,}")

print("Packet and byte summary:")
display(
    zero_duration_flows[["Total Packets", "Total Bytes"]].describe(
        percentiles=[0.01, 0.25, 0.5, 0.75, 0.95, 0.99]
    )
)

packet_buckets = pd.cut(
    zero_duration_flows["Total Packets"],
    bins=[-np.inf, 0, 1, 2, 10, 100, np.inf],
    labels=["0 or fewer", "1", "2", "3-10", "11-100", "more than 100"],
)
packet_bucket_summary = (
    packet_buckets.value_counts(sort=False)
    .rename("flow_count")
    .to_frame()
)
packet_bucket_summary["percentage"] = (
    packet_bucket_summary["flow_count"] / len(zero_duration_flows) * 100
)
print("Total-packet distribution:")
display(packet_bucket_summary)

basic_consistency_summary = pd.Series(
    {
        "zero or negative total packets": int(
            zero_duration_flows["Total Packets"].le(0).sum()
        ),
        "negative total bytes": int(
            zero_duration_flows["Total Bytes"].lt(0).sum()
        ),
        "zero total bytes": int(
            zero_duration_flows["Total Bytes"].eq(0).sum()
        ),
        "more than 10 packets": int(
            zero_duration_flows["Total Packets"].gt(10).sum()
        ),
        "more than 100 packets": int(
            zero_duration_flows["Total Packets"].gt(100).sum()
        ),
    },
    name="flow_count",
).to_frame()
print("Basic consistency checks:")
display(basic_consistency_summary)

iat_missing_counts = zero_duration_flows[iat_features].isna().sum()
iat_nonzero_counts = (
    zero_duration_flows[iat_features].notna()
    & zero_duration_flows[iat_features].ne(0)
).sum()
iat_consistency_summary = pd.DataFrame(
    {
        "missing_count": iat_missing_counts,
        "nonzero_count": iat_nonzero_counts,
        "minimum": zero_duration_flows[iat_features].min(),
        "maximum": zero_duration_flows[iat_features].max(),
    }
)
iat_consistency_summary = iat_consistency_summary.loc[
    (iat_consistency_summary["missing_count"] > 0)
    | (iat_consistency_summary["nonzero_count"] > 0)
]
print("IAT features that are missing or non-zero:")
if iat_consistency_summary.empty:
    print("All IAT features are present and equal to zero.")
else:
    display(iat_consistency_summary)

protocol_summary = (
    zero_duration_flows.groupby("Protocol", dropna=False)
    .agg(
        flow_count=("Protocol", "size"),
        median_packets=("Total Packets", "median"),
        maximum_packets=("Total Packets", "max"),
        median_bytes=("Total Bytes", "median"),
        maximum_bytes=("Total Bytes", "max"),
    )
    .sort_values("flow_count", ascending=False)
)
protocol_summary["percentage"] = (
    protocol_summary["flow_count"] / len(zero_duration_flows) * 100
)
print("Protocol distribution:")
display(protocol_summary)

zero_duration_label_summary = (
    zero_duration_flows["Label"]
    .value_counts()
    .rename("flow_count")
    .to_frame()
)
zero_duration_label_summary["percentage"] = (
    zero_duration_label_summary["flow_count"]
    / len(zero_duration_flows)
    * 100
)
print("Label distribution:")
display(zero_duration_label_summary)

largest_zero_duration_columns = [
    "Flow ID",
    "Src IP",
    "Src Port",
    "Dst IP",
    "Dst Port",
    "Protocol",
    "Timestamp",
    "Label",
    "Total Fwd Packet",
    "Total Bwd packets",
    "Total Packets",
    "Total Length of Fwd Packet",
    "Total Length of Bwd Packet",
    "Total Bytes",
    "Flow IAT Min",
    "Flow IAT Max",
    "Fwd Header Length",
    "Bwd Header Length",
]
print("Twenty zero-duration flows with the most packets:")
display(
    zero_duration_flows.nlargest(20, "Total Packets")[
        largest_zero_duration_columns
    ]
)

Zero-duration rows under investigation: 2,866
Packet and byte summary:


,Total Packets,Total Bytes
count,2866.000000,2866.000000
mean,2.000349,9.022331
std,0.018679,50.333369
min,2.000000,0.000000
1%,2.000000,0.000000
25%,2.000000,0.000000
50%,2.000000,4.000000
75%,2.000000,12.000000
95%,2.000000,31.000000
99%,2.000000,40.400000


Total-packet distribution:


,flow_count,percentage
Total Packets,,
0 or fewer,0,0.000000
1,0,0.000000
2,2865,99.965108
3-10,1,0.034892
11-100,0,0.000000
more than 100,0,0.000000


Basic consistency checks:


...

,flow_count,median_packets,maximum_packets,median_bytes,maximum_bytes,percentage
Protocol,,,,,,
6.0,2849,2.0,3.0,4.0,2065.0,99.406839
17.0,17,2.0,2.0,54.0,124.0,0.593161


Label distribution:


,flow_count,percentage
Label,,
BENIGN,1776,61.967900
DoS Hulk,949,33.112352
PortScan,126,4.396371
Bot,10,0.348918
FTP-Patator,3,0.104676
DDoS,2,0.069784


Twenty zero-duration flows with the most packets:


,Flow ID,Src IP,Src Port,Dst IP,Dst Port,Protocol,Timestamp,Label,Total Fwd Packet,Total Bwd packets,Total Packets,Total Length of Fwd Packet,Total Length of Bwd Packet,Total Bytes,Flow IAT Min,Flow IAT Max,Fwd Header Length,Bwd Header Length
1269336,192.168.10.8-192.168.10.50-53335-22-6,192.168.10.50,22.0,192.168.10.8,53335.0,6.0,6/7/2017 1:45,BENIGN,1.0,2.0,3.0,6.0,12.0,18.0,0.0,0.0,20.0,40.0
65,185.86.137.17-192.168.10.5-443-55043-6,185.86.137.17,443.0,192.168.10.5,55043.0,6.0,7/7/2017 3:30,BENIGN,2.0,0.0,2.0,12.0,0.0,12.0,0.0,0.0,40.0,0.0
1763,149.174.66.134-192.168.10.16-443-60018-6,149.174.66.134,443.0,192.168.10.16,60018.0,6.0,7/7/2017 3:31,BENIGN,2.0,0.0,2.0,12.0,0.0,12.0,0.0,0.0,40.0,0.0
1886,172.217.12.142-192.168.10.14-80-57855-6,192.168.10.14,57855.0,172.217.12.142,80.0,6.0,7/7/2017 3:31,BENIGN,1.0,1.0,2.0,6.0,6.0,12.0,0.0,0.0,20.0,20.0
3368,192.168.10.5-54.76.211.99-55251-443-6,192.168.10.5,55251.0,54.76.211.99,443.0,6.0,7/7/2017 3:31,BENIGN,1.0,1.0,2.0,6.0,6.0,12.0,0.0,0.0,20.0,20.0
6789,192.168.10.16-198.54.12.145-36812-80-6,198.54.12.145,80.0,192.168.10.16,36812.0,6.0,7/7/2017 3:35,BENIGN,2.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,64.0,0.0
8048,192.168.10.3-192.168.10.19-3268-35382-6,192.168.10.3,3268.0,192.168.10.19,35382.0,6.0,7/7/2017 3:37,BENIGN,1.0,1.0,2.0,6.0,0.0,6.0,0.0,0.0,20.0,32.0
8396,192.168.10.5-192.168.10.50-55421-21-6,192.168.10.50,21.0,192.168.10.5,55421.0,6.0,7/7/2017 3:37,BENIGN,2.0,0.0,2.0,20.0,0.0,20.0,0.0,0.0,40.0,0.0
13302,192.168.10.9-52.84.145.229-9242-443-6,52.84.145.229,443.0,192.168.10.9,9242.0,6.0,7/7/2017 3:42,BENIGN,2.0,0.0,2.0,12.0,0.0,12.0,0.0,0.0,40.0,0.0
13705,192.168.10.3-192.168.10.16-389-50546-6,192.168.10.16,50546.0,192.168.10.3,389.0,6.0,7/7/2017 3:45,BENIGN,2.0,0.0,2.0,7.0,0.0,7.0,0.0,0.0,64.0,0.0


### Interpretation of the zero-duration flows

The remaining 2,866 zero-duration rows are internally plausible very short flows rather than obvious corruption:

- 2,865 contain exactly two packets and one contains three; none contains more than three packets.
- Every IAT feature is present and equal to zero.
- None has zero/negative packet counts or negative transferred bytes.
- 2,849 are TCP and 17 are UDP. Zero payload bytes in 1,357 rows can represent TCP control packets and is not itself invalid.
- Missing `Flow Bytes/s` and `Flow Packets/s` values match the zero-duration rows exactly.

These rows could be kept as timestamp-resolution artifacts, but both rate features are mathematically undefined for them. For a simple first benchmark, removing these approximately 0.1% of rows is the cleanest option: it preserves the two rate features and removes the remaining missing values without inventing rates. The next section applies that baseline choice. A later experiment can compare it with keeping the rows, adding a zero-duration indicator, and using model-aware missing-value handling.

## 23. Remove zero-duration flows

Remove the investigated zero-duration rows for the first benchmark. Before removal, assert that they still match the missing values in both rate columns exactly. This guards against accidentally deleting rows for unrelated missing values.

In [23]:
zero_duration_removal_mask = df_valid["Flow Duration"].eq(0)
both_rate_features_missing_mask = (
    df_valid["Flow Bytes/s"].isna()
    & df_valid["Flow Packets/s"].isna()
)

if not zero_duration_removal_mask.equals(both_rate_features_missing_mask):
    raise ValueError(
        "Zero-duration rows no longer match rows missing both rate features. "
        "Investigate before removing them."
    )

zero_duration_removed_by_label = (
    df_valid.loc[zero_duration_removal_mask, "Label"]
    .value_counts()
    .rename("removed_rows")
    .to_frame()
)

rows_before_zero_duration_removal = len(df_valid)
zero_duration_rows_removed = int(zero_duration_removal_mask.sum())
df_valid.drop(
    index=df_valid.index[zero_duration_removal_mask],
    inplace=True,
)
df_valid.reset_index(drop=True, inplace=True)

remaining_missing_values = df_valid.isna().sum()
remaining_missing_values = remaining_missing_values[
    remaining_missing_values > 0
].sort_values(ascending=False)

print(f"Rows before removal: {rows_before_zero_duration_removal:,}")
print(f"Zero-duration rows removed: {zero_duration_rows_removed:,}")
print(f"Rows remaining: {len(df_valid):,}")
print(f"Remaining missing cells: {int(df_valid.isna().sum().sum()):,}")
display(zero_duration_removed_by_label)

if remaining_missing_values.empty:
    print("No missing feature values remain.")
else:
    display(remaining_missing_values.rename("missing_count").to_frame())

Rows before removal: 2,827,817
Zero-duration rows removed: 2,866
Rows remaining: 2,824,951


Remaining missing cells: 0


,removed_rows
Label,
BENIGN,1776
DoS Hulk,949
PortScan,126
Bot,10
FTP-Patator,3
DDoS,2


No missing feature values remain.


## 24. Duplicate records and label conflicts

Group rows using every remaining column except `Label`. A group with multiple rows and one label represents exact repeated records; a group with multiple labels represents identical recorded features with a target conflict. Exact same-label copies can be removed directly, while conflicting-label rows require investigation. The summary provides an audit trail of what will be removed and ensures these two cases are not mixed.

In [24]:
duplicate_feature_columns = [
    column for column in df_valid.columns if column != "Label"
]

feature_fingerprint_1 = pd.util.hash_pandas_object(
    df_valid[duplicate_feature_columns],
    index=False,
    hash_key="0123456789abcdef",
)
feature_fingerprint_2 = pd.util.hash_pandas_object(
    df_valid[duplicate_feature_columns],
    index=False,
    hash_key="fedcba9876543210",
)

fingerprint_columns = ["_fingerprint_1", "_fingerprint_2"]
duplicate_audit = pd.DataFrame(
    {
        "_row_index": df_valid.index,
        "_fingerprint_1": feature_fingerprint_1.to_numpy(),
        "_fingerprint_2": feature_fingerprint_2.to_numpy(),
        "Label": df_valid["Label"].to_numpy(),
    },
    index=df_valid.index,
)

feature_group_summary = (
    duplicate_audit.groupby(fingerprint_columns, sort=False, observed=True)
    .agg(
        row_count=("_row_index", "size"),
        unique_label_count=("Label", "nunique"),
    )
)

exact_duplicate_group_summary = feature_group_summary.loc[
    (feature_group_summary["row_count"] > 1)
    & (feature_group_summary["unique_label_count"] == 1)
]
conflicting_label_group_summary = feature_group_summary.loc[
    feature_group_summary["unique_label_count"] > 1
]

exact_duplicate_group_count = len(exact_duplicate_group_summary)
duplicate_row_count = int(
    exact_duplicate_group_summary["row_count"].sum()
)
removable_extra_copy_count = int(
    (exact_duplicate_group_summary["row_count"] - 1).sum()
)
conflicting_label_group_count = len(conflicting_label_group_summary)
rows_in_conflicting_label_groups = int(
    conflicting_label_group_summary["row_count"].sum()
)

rows_after_duplicate_removal = (
    len(df_valid) - removable_extra_copy_count
)
duplicate_overview = pd.DataFrame(
    {
        "Count": [
            len(df_valid),
            rows_after_duplicate_removal,
            duplicate_row_count,
            exact_duplicate_group_count,
            removable_extra_copy_count,
            conflicting_label_group_count,
            rows_in_conflicting_label_groups,
        ],
    },
    index=[
        "Rows before duplicate removal",
        "Rows after duplicate removal",
        "Duplicate rows",
        "Distinct Duplicate Rows",
        "Removable extra copies",
        "Conflicting-label groups",
        "Rows in conflicting-label groups",
    ],
)

display(duplicate_overview)
print(
    "Percentage of rows belonging to same-label duplicate groups: "
    f"{duplicate_row_count / len(df_valid) * 100:.6f}%"
)
print(
    "Percentage represented by removable extra copies: "
    f"{removable_extra_copy_count / len(df_valid) * 100:.6f}%"
)
print(
    "Maximum exact-duplicate group size:",
    int(exact_duplicate_group_summary["row_count"].max())
    if not exact_duplicate_group_summary.empty
    else 1,
)
print(
    "Maximum conflicting-label group size:",
    int(conflicting_label_group_summary["row_count"].max())
    if not conflicting_label_group_summary.empty
    else 0,
)

,Count
Rows before duplicate removal,2824951
Rows after duplicate removal,2824752
Duplicate rows,301
Distinct Duplicate Rows,102
Removable extra copies,199
Conflicting-label groups,0
Rows in conflicting-label groups,0


Percentage of rows belonging to same-label duplicate groups: 0.010655%
Percentage represented by removable extra copies: 0.007044%
Maximum exact-duplicate group size: 13
Maximum conflicting-label group size: 0


The 2,824,951 cleaned rows become 2,824,752 rows after duplicate removal. There are 301 duplicate rows across 102 distinct duplicated records; keeping one occurrence of each record leaves 199 removable extra copies. Those extra copies represent only `0.007%` of the dataset, the largest group contains 13 rows, and no group contains conflicting labels. Therefore duplication is rare rather than a dataset-wide pattern, and the target is consistent wherever the complete recorded feature vector repeats.

### 24.1 Exact same-label duplicate details

In [25]:
duplicate_group_size_counts = (
    exact_duplicate_group_summary["row_count"].value_counts().sort_index()
)
exact_duplicate_size_distribution = pd.DataFrame(
    {
        "Number of duplicate groups": duplicate_group_size_counts.to_numpy(),
        "Total rows in same-label duplicate groups": (
            duplicate_group_size_counts.index.to_numpy()
            * duplicate_group_size_counts.to_numpy()
        ),
        "Removable extra copies": (
            (duplicate_group_size_counts.index.to_numpy() - 1)
            * duplicate_group_size_counts.to_numpy()
        ),
    },
    index=pd.Index(
        duplicate_group_size_counts.index,
        name="Rows per duplicate group",
    ),
)
print("Exact-duplicate group-size distribution:")
display(exact_duplicate_size_distribution)

exact_duplicate_key_table = (
    exact_duplicate_group_summary.reset_index()[fingerprint_columns]
)
exact_duplicate_rows = duplicate_audit.merge(
    exact_duplicate_key_table,
    on=fingerprint_columns,
    how="inner",
)
exact_duplicate_rows["_copy_number"] = (
    exact_duplicate_rows.groupby(fingerprint_columns, sort=False).cumcount()
)
removable_extra_rows = exact_duplicate_rows.loc[
    exact_duplicate_rows["_copy_number"] > 0
]
removable_extra_copy_label_distribution = (
    removable_extra_rows["Label"]
    .value_counts()
    .rename("Removable extra copies")
    .to_frame()
)
removable_extra_copy_label_distribution[
    "Percentage of removable extra copies"
] = (
    removable_extra_copy_label_distribution["Removable extra copies"]
    / len(removable_extra_rows)
    * 100
    if len(removable_extra_rows)
    else 0
)
print("Removable extra copies by label:")
display(removable_extra_copy_label_distribution)

duplicate_example_columns = [
    "Flow ID",
    "Src IP",
    "Src Port",
    "Dst IP",
    "Dst Port",
    "Protocol",
    "Timestamp",
    "Flow Duration",
    "Total Fwd Packet",
    "Total Bwd packets",
    "Total Length of Fwd Packet",
    "Total Length of Bwd Packet",
    "Label",
]

if exact_duplicate_group_summary.empty:
    print("No exact same-label duplicate groups found.")
else:
    duplicate_example_frames = []
    for group_number, group_key in enumerate(
        exact_duplicate_group_summary.head(5).index,
        start=1,
    ):
        group_rows = exact_duplicate_rows.loc[
            (exact_duplicate_rows["_fingerprint_1"] == group_key[0])
            & (exact_duplicate_rows["_fingerprint_2"] == group_key[1])
        ]
        example = df_valid.loc[
            group_rows["_row_index"], duplicate_example_columns
        ].copy()
        example.insert(0, "Duplicate group", group_number)
        duplicate_example_frames.append(example)

    print("Representative exact-duplicate groups:")
    display(pd.concat(duplicate_example_frames))

Exact-duplicate group-size distribution:


,Number of duplicate groups,Total rows in same-label duplicate groups,Removable extra copies
Rows per duplicate group,,,
2,49,98,49
3,33,99,66
4,11,44,33
5,5,25,20
6,1,6,5
8,2,16,14
13,1,13,12


Removable extra copies by label:


,Removable extra copies,Percentage of removable extra copies
Label,,
BENIGN,198,99.497487
DoS Hulk,1,0.502513


Representative exact-duplicate groups:


,Duplicate group,Flow ID,Src IP,Src Port,Dst IP,Dst Port,Protocol,Timestamp,Flow Duration,Total Fwd Packet,Total Bwd packets,Total Length of Fwd Packet,Total Length of Bwd Packet,Label
17618,1,192.168.10.25-192.168.10.50-53615-6756-6,192.168.10.25,53615.0,192.168.10.50,6756.0,6.0,7/7/2017 3:53,1.0,2.0,0.0,0.0,0.0,BENIGN
17619,1,192.168.10.25-192.168.10.50-53615-6756-6,192.168.10.25,53615.0,192.168.10.50,6756.0,6.0,7/7/2017 3:53,1.0,2.0,0.0,0.0,0.0,BENIGN
59664,2,192.168.10.25-192.168.10.50-53631-20878-6,192.168.10.25,53631.0,192.168.10.50,20878.0,6.0,7/7/2017 4:00,1.0,2.0,0.0,0.0,0.0,BENIGN
59665,2,192.168.10.25-192.168.10.50-53631-20878-6,192.168.10.25,53631.0,192.168.10.50,20878.0,6.0,7/7/2017 4:00,1.0,2.0,0.0,0.0,0.0,BENIGN
493604,3,192.168.10.25-192.168.10.50-53461-6328-6,192.168.10.25,53461.0,192.168.10.50,6328.0,6.0,7/7/2017 3:22,1.0,2.0,0.0,0.0,0.0,BENIGN
493608,3,192.168.10.25-192.168.10.50-53461-6328-6,192.168.10.25,53461.0,192.168.10.50,6328.0,6.0,7/7/2017 3:22,1.0,2.0,0.0,0.0,0.0,BENIGN
665765,4,192.168.10.25-192.168.10.50-51807-16028-6,192.168.10.25,51807.0,192.168.10.50,16028.0,6.0,7/7/2017 11:32,1.0,2.0,0.0,0.0,0.0,BENIGN
665768,4,192.168.10.25-192.168.10.50-51807-16028-6,192.168.10.25,51807.0,192.168.10.50,16028.0,6.0,7/7/2017 11:32,1.0,2.0,0.0,0.0,0.0,BENIGN
687948,5,192.168.10.25-192.168.10.50-52265-44410-6,192.168.10.25,52265.0,192.168.10.50,44410.0,6.0,7/7/2017 11:59,1.0,2.0,0.0,0.0,0.0,BENIGN
687951,5,192.168.10.25-192.168.10.50-52265-44410-6,192.168.10.25,52265.0,192.168.10.50,44410.0,6.0,7/7/2017 11:59,1.0,2.0,0.0,0.0,0.0,BENIGN


Most duplicate groups are small: 49 contain two rows and 33 contain three. Of the 199 removable extra copies, 198 are `BENIGN` and one is `DoS Hulk`. The displayed examples repeat the same flow ID, timestamp, endpoints, packet counts and measurements, which is stronger evidence of duplicated records than merely similar traffic. Keeping one row from each group will have a negligible effect on class balance while preventing identical records from later appearing in different data splits.

### 24.2 Identical features with conflicting labels

In [26]:
conflicting_label_key_table = (
    conflicting_label_group_summary.reset_index()[fingerprint_columns]
)
conflicting_label_rows = duplicate_audit.merge(
    conflicting_label_key_table,
    on=fingerprint_columns,
    how="inner",
)

if conflicting_label_group_summary.empty:
    print("No identical feature vectors with conflicting labels found.")
else:
    conflicting_label_combinations = (
        conflicting_label_rows.groupby(fingerprint_columns, sort=False)[
            "Label"
        ]
        .agg(lambda labels: " | ".join(sorted(set(labels))))
        .value_counts()
        .rename_axis("Conflicting label combination")
        .rename("Number of feature groups")
        .to_frame()
    )
    print("Conflicting label combinations:")
    display(conflicting_label_combinations)

    conflict_example_frames = []
    for group_number, group_key in enumerate(
        conflicting_label_group_summary.head(5).index,
        start=1,
    ):
        group_rows = conflicting_label_rows.loc[
            (conflicting_label_rows["_fingerprint_1"] == group_key[0])
            & (conflicting_label_rows["_fingerprint_2"] == group_key[1])
        ]
        example = df_valid.loc[
            group_rows["_row_index"], duplicate_example_columns
        ].copy()
        example.insert(0, "Conflicting group", group_number)
        conflict_example_frames.append(example)

    print("Representative conflicting-label groups:")
    display(pd.concat(conflict_example_frames))

No identical feature vectors with conflicting labels found.


No complete feature vector is associated with more than one label. This rules out direct target contradictions at the current schema level, so no conflict-resolution policy is needed. The check should be repeated in the feature-engineering notebook after identifiers are removed, because dropping columns can make previously distinct rows collapse to the same model input.

## 25. Remove confirmed duplicates and validate the final dataset

The evidence in section 24 supports keeping the first occurrence of each exact record and removing only its removable extra copies. Rows with different labels would not be removed by this operation, although section 24 found no such conflicts.

In [27]:
expected_removable_extra_copies = removable_extra_copy_count
exact_duplicate_removal_mask = df_valid.duplicated(keep="first")
exact_duplicate_rows_to_remove = int(exact_duplicate_removal_mask.sum())

if exact_duplicate_rows_to_remove != expected_removable_extra_copies:
    raise ValueError(
        "Exact duplicate count changed between investigation and removal: "
        f"expected {expected_removable_extra_copies:,}, "
        f"found {exact_duplicate_rows_to_remove:,}."
    )

removed_duplicate_label_distribution = (
    df_valid.loc[exact_duplicate_removal_mask, "Label"]
    .value_counts()
    .rename("Removable extra copies")
    .to_frame()
)
removed_duplicate_label_distribution[
    "Percentage of removable extra copies"
] = (
    removed_duplicate_label_distribution["Removable extra copies"]
    / exact_duplicate_rows_to_remove
    * 100
    if exact_duplicate_rows_to_remove
    else 0
)

rows_before_duplicate_removal = len(df_valid)
df_valid.drop(
    index=df_valid.index[exact_duplicate_removal_mask],
    inplace=True,
)
df_valid.reset_index(drop=True, inplace=True)

print(f"Rows before removal: {rows_before_duplicate_removal:,}")
print(f"Removable extra copies removed: {exact_duplicate_rows_to_remove:,}")
print(f"Rows remaining: {len(df_valid):,}")
display(removed_duplicate_label_distribution)

Rows before removal: 2,824,951
Removable extra copies removed: 199
Rows remaining: 2,824,752


,Removable extra copies,Percentage of removable extra copies
Label,,
BENIGN,198,99.497487
DoS Hulk,1,0.502513


The removal count exactly matches the 199 removable extra copies identified in section 24, confirming that no additional records were deleted. Removing 198 benign rows and one `DoS Hulk` row changes neither the overall class structure nor the rare classes in a meaningful way. The dataset now contains 2,824,752 unique complete records.

In [28]:
final_numeric_features = df_valid.select_dtypes(include="number").columns.tolist()
final_window_features = ["FWD Init Win Bytes", "Bwd Init Win Bytes"]
final_nonnegative_features = [
    feature
    for feature in final_numeric_features
    if feature not in final_window_features
]

final_missing_cells = int(df_valid.isna().sum().sum())
final_infinite_cells = int(
    np.isinf(df_valid[final_numeric_features]).sum().sum()
)
final_nonpositive_duration_rows = int(df_valid["Flow Duration"].le(0).sum())
final_unexpected_negative_cells = int(
    df_valid[final_nonnegative_features].lt(0).sum().sum()
)
final_invalid_window_cells = int(
    (
        df_valid[final_window_features].lt(-1)
        | df_valid[final_window_features].gt(65535)
    )
    .sum()
    .sum()
)
final_exact_duplicate_rows = int(df_valid.duplicated(keep=False).sum())
final_feature_columns = [
    column for column in df_valid.columns if column != "Label"
]
final_conflicting_feature_rows = int(
    df_valid.duplicated(
        subset=final_feature_columns,
        keep=False,
    ).sum()
)

final_dataset_dimensions = pd.DataFrame(
    {"Count": [len(df_valid), df_valid.shape[1]]},
    index=["Rows after all cleaning", "Columns after all cleaning"],
)
final_integrity_checks = pd.DataFrame(
    {
        "Observed count": [
            int(df_valid["Label"].isna().sum()),
            final_missing_cells,
            final_infinite_cells,
            final_nonpositive_duration_rows,
            final_unexpected_negative_cells,
            final_invalid_window_cells,
            final_exact_duplicate_rows,
            final_conflicting_feature_rows,
        ],
        "Expected count": [0] * 8,
    },
    index=[
        "Missing labels",
        "Missing cells (entire dataset)",
        "Infinite numeric cells",
        "Rows with zero or negative duration",
        "Unexpected negative numeric cells",
        "Invalid TCP-window cells",
        "Rows belonging to exact-duplicate groups",
        "Rows belonging to conflicting-label groups",
    ],
)
display(final_dataset_dimensions)
display(final_integrity_checks)

validation_checks = final_integrity_checks["Observed count"]
if validation_checks.ne(0).any():
    raise AssertionError(
        "Final validation failed:\n"
        + validation_checks[validation_checks.ne(0)].to_string()
    )

final_label_distribution = (
    df_valid["Label"]
    .value_counts()
    .rename("Flow count")
    .to_frame()
)
final_label_distribution["Percentage of dataset"] = (
    final_label_distribution["Flow count"] / len(df_valid) * 100
)
display(final_label_distribution)
print("All final data-integrity checks passed.")

,Count
Rows after all cleaning,2824752
Columns after all cleaning,76


,Observed count,Expected count
Missing labels,0,0
Missing cells (entire dataset),0,0
Infinite numeric cells,0,0
Rows with zero or negative duration,0,0
Unexpected negative numeric cells,0,0
Invalid TCP-window cells,0,0
Rows belonging to exact-duplicate groups,0,0
Rows belonging to conflicting-label groups,0,0


,Flow count,Percentage of dataset
Label,,
BENIGN,2268391,80.304076
DoS Hulk,229964,8.141033
PortScan,158804,5.621874
DDoS,128006,4.531584
DoS GoldenEye,10288,0.364209
FTP-Patator,7931,0.280768
SSH-Patator,5895,0.208691
DoS slowloris,5796,0.205186
DoS Slowhttptest,5499,0.194672


All final data-integrity checks passed.


Every final integrity counter is zero: there are no missing labels or feature values, infinities, nonpositive durations, unexpected negatives, invalid TCP-window values, exact duplicates or complete-feature label conflicts. This confirms that row-level cleaning is complete. The remaining issue is modeling difficulty rather than data corruption: `BENIGN` still represents `80.30%` of flows, while Heartbleed has only 7 rows, SQL Injection 21 and Infiltration 35. The next stage must preserve these rare classes during splitting and use metrics suited to multiclass imbalance.

## 26. Inspect final Protocol values

Display the unmodified `Protocol.value_counts()` result after all cleaning is complete. Assign readable names only after confirming the values present in the exact dataset that will be saved. This section does not change the `Protocol` column or its dtype.

In [29]:
raw_protocol_counts = (
    df_valid["Protocol"].value_counts(dropna=False).sort_index()
)
raw_protocol_distribution = (
    raw_protocol_counts
    .rename_axis("Raw Protocol value")
    .rename("Flow count")
    .to_frame()
)
raw_protocol_distribution["Percentage of dataset"] = (
    raw_protocol_distribution["Flow count"] / len(df_valid) * 100
)
display(raw_protocol_distribution)

,Flow count,Percentage of dataset
Raw Protocol value,,
0.0,1696,0.060041
6.0,1823631,64.558977
17.0,999425,35.380982


The observed protocol values and their names are:

- `0`: HOPOPT
- `6`: TCP
- `17`: UDP

The final dataset retains the numeric `Protocol` values unchanged. Their representation for particular model families remains a later modeling decision.

## 27. Save the cleaned dataset

Save the final validated working dataset as Parquet so later notebooks can load it directly without repeating the complete cleaning process.

In [30]:
PROCESSED_DATA_PATH = (
    DATASET_PATH.parent.parent
    / "processed"
    / "cicids2017_cleaned.parquet"
)
PROCESSED_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)

df_valid.to_parquet(PROCESSED_DATA_PATH, index=False)

if not PROCESSED_DATA_PATH.is_file():
    raise FileNotFoundError("The cleaned Parquet file was not created.")

print(f"Saved cleaned dataset: {PROCESSED_DATA_PATH}")
print(f"Rows saved: {len(df_valid):,}")
print(f"Columns saved: {df_valid.shape[1]}")
print(
    "File size: "
    f"{PROCESSED_DATA_PATH.stat().st_size / (1024 ** 2):,.2f} MiB"
)

Saved cleaned dataset: C:\Users\ademz\Desktop\9raya\DoS-Intrusion-Detection-System\ml\data\processed\cicids2017_cleaned.parquet
Rows saved: 2,824,752
Columns saved: 76
File size: 360.48 MiB


## Final EDA summary

- The raw merged dataset contained 3,119,345 rows and 84 columns.
- 288,602 completely empty records were removed, leaving 2,830,743 labeled rows for validation.
- Infinite rate values were normalized to missing values before their cause was investigated.
- 2,926 rows with impossible negative timing, header or segment measurements were removed.
- The remaining 2,866 zero-duration flows were internally plausible short flows, but their two rate features were undefined; removing them eliminated all missing values without imputation.
- Eight exact constant columns were removed. Nonconstant semantically related columns were retained for later correlation and feature selection.
- Section 10 normalized the three malformed `Web Attack` labels without changing their classes or counts.
- Section 24 found 301 duplicate rows across 102 distinct duplicated records and no conflicting-label groups. Keeping one occurrence of each record removed 199 removable extra copies.
- The final working dataset contains **2,824,752 rows and 76 columns**.
- Final validation found no missing labels, missing feature values, infinities, nonpositive durations, unexpected negative values, invalid TCP-window values, exact duplicates or complete-feature label conflicts.
- The target contains 15 classes and remains highly imbalanced: `BENIGN` accounts for 80.30%, while Heartbleed has 7 rows, SQL Injection 21 and Infiltration 35.
- The final `Protocol` values are `0` (HOPOPT), `6` (TCP), and `17` (UDP); the stored numeric column remains unchanged.
- Section 27 saved the final dataset to `ml/data/processed/cicids2017_cleaned.parquet`.

The data-quality EDA is complete. Feature preparation, leakage-prone identifier decisions, splitting and train-fitted feature selection belong in the next notebook.